# Luma Phase 2 Module Implementation
This notebook contain the implementation of phase 2 of Luma Geospatial Engine

## Prerequisite
Earth engine initialization using service account

In [1]:
import ee
ee.Authenticate()
ee.Initialize()


## Retrieve AOI from Earth Engine Asset Manager

In [2]:
from luma_ge.data_acquisition import GEE_Asset_Manager
#Initialize the Asset Manager from Earth Engine
asset = GEE_Asset_Manager()
if asset.load_asset():
    regency_names = asset.get_regency_names()
    if regency_names:
        print(f"✓ Found {len(regency_names)} regencies")
        #use city name
        selected_regency = "Kota Bandung" 
        #Verify the city name exists in the regency list
        if selected_regency in regency_names:
            print(f"\n✓ Selected regency: {selected_regency}")
        else:
            print(f"\n✗ Regency '{selected_regency}' not found")
            print(f"\nAvailable regencies:")
            for i, name in enumerate(regency_names, 1):
                print(f"  {i}. {name}")
            selected_regency = None
    else:
        print("✗ No regency names found")
        selected_regency = None
else:
    print("✗ Failed to load asset")
    selected_regency = None
#Retrieve AOI geometry for the selected regency
if selected_regency:
    aoi = asset.get_regency_geometry(selected_regency)
    
    if aoi:
        print(f"✓ Successfully retrieved geometry for: {selected_regency}")
    else:
        print(f"✗ Failed to retrieve geometry for: {selected_regency}")
        aoi = None
else:
    print("✗ No regency selected")
    aoi = None

2026-06-03 16:07:06,926 - luma_ge.data_acquisition - INFO - Successfully loaded asset with 548 features
2026-06-03 16:07:07,365 - luma_ge.data_acquisition - INFO - Extracted 516 unique regency names


✓ Found 516 regencies

✓ Selected regency: Kota Bandung


2026-06-03 16:07:07,833 - luma_ge.data_acquisition - INFO - Successfully retrieved geometry for: Kota Bandung


✓ Successfully retrieved geometry for: Kota Bandung


## New Feature: Sentinel-2 Data Retrieval



In [7]:
from luma_ge.data_acquisition import Reflectance_Data, final_Image
import geemap
#initialize class
optical_reflectance = Reflectance_Data()
composite = final_Image()
#define the temporal range
start = '2024-01-01'
end = '2024-12-30'
#retrieve the sentinel 2 data
s2_data, metas2 = optical_reflectance.get_s2_optical_data(aoi, start, end, cloud_cover=20, compute_detailed_stats=False)
#final output: composite image
median_s2 = composite.get_temporal_composite(s2_data, aoi, calculate_coverage=False, coverage_scale=10)
#10m band
visnir = median_s2.select(['RED', 'GREEN', 'BLUE', 'NIR'])
#20m band
reswir = median_s2.select(['RED_EDGE1', 'RED_EDGE2', 'RED_EDGE3', 'RED_EDGE4', 'SWIR1', 'SWIR2'])
vis_10m = {'min': 0,'max': 0.3,'gamma': [0.95, 1.1, 1],'bands':['RED', 'GREEN', 'BLUE']}
res_20m = {'min': 0,'max': 0.3,'gamma': [0.95, 1.1, 1],'bands':['RED', 'GREEN', 'BLUE']}
Map = geemap.Map()
Map.addLayer(median_s2, vis_10m, 'Sentinel-2 Composite 10 m band')
Map.addLayer(median_s2, res_20m, 'Sentinel-2 Composite 20 m band')
Map.centerObject(aoi, 9)
Map

2026-06-03 16:36:37,847 - Reflectance_Data - INFO - ReflectanceData initialized.
2026-06-03 16:36:37,849 - final_Image - INFO - final_Image creation initialized.
2026-06-03 16:36:37,850 - Reflectance_Data - INFO - Starting data fetch for Sentinel-2 Level-2A Surface Reflectance (Harmonized)
2026-06-03 16:36:37,851 - Reflectance_Data - INFO - Date range: 2024-01-01 to 2024-12-30
2026-06-03 16:36:37,853 - Reflectance_Data - INFO - Cloud cover threshold (image-level): 20%
2026-06-03 16:36:37,855 - Reflectance_Data - INFO - Cloud Score+ pixel threshold: 0.6
2026-06-03 16:36:37,856 - Reflectance_Data - INFO - Detailed statistics will not be computed
2026-06-03 16:36:37,858 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-06-03 16:36:37,865 - Reflectance_Data - INFO - Filtered collection created (use compute_detailed_stats=True for more information)
2026-06-03 16:36:38,520 - final_Image - INFO - Creating median composite from 66 images
2026-06-03 16:36:38,522 - final_Image - I

Map(center=[-6.919241950180151, 107.63659926544979], controls=(WidgetControl(options=['position', 'transparent…

In [4]:
# Apply sharpening to the Sentinel-2 composite
sharpened_s2 = optical_reflectance.sharpen_s2_bands(median_s2, aoi=aoi)


if sharpened_s2 is not None:
    print("Sharpening applied successfully.")
    # Visualize sharpened RGB
    vis_sharp = {'min': 0, 'max': 0.3, 'gamma': [0.95, 1.1, 1], 'bands': ['RED_EDGE1', 'RED_EDGE2', 'RED_EDGE3']}
    Map.addLayer(sharpened_s2, vis_sharp, 'Sentinel-2 Sharpened RGB')
    Map.centerObject(aoi, 9)
    Map
else:
    print("Sharpening failed. Check that the input image has standardized Sentinel-2 band names.")
    

Sharpening applied successfully.


In [ ]:
# Sanity check — run this BEFORE calling sharpen_s2_bands
print(type(median_s2))                          # should be ee.Image
print(median_s2.bandNames().getInfo())          # should list your renamed bands
print(median_s2.select('NIR').projection().getInfo())  # should show a real CRS

In [9]:
#print(sharpened_s2.bandNames().getInfo())
re1 = sharpened_s2.select('RED_EDGE1')
m = geemap.Map()
m.addLayer(sharpened_s2, res_20m, 'SharpenedBand')
m.addLayer(re1, {}, 'Red Edge Sharpen')
m

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [5]:
print(type(sharpened_s2))   

<class 'ee.image.Image'>


In [6]:
export_task = ee.batch.Export.image.toDrive(
     image=sharpened_s2,
     description='Bandung_S2_sharpen',
     folder='Earth Engine',
     fileNamePrefix='Bandung_S2_sharpen',
     scale=10,
     region=aoi,  # or aoi.geometry()
     maxPixels=1e13
 )
export_task.start()
import time

while export_task.active():
     print('Exporting... (status: {})'.format(export_task.status()['state']))
     time.sleep(10)

print('Export complete (status: {})'.format(export_task.status()['state']))

Exporting... (status: READY)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting... (status: RUNNING)
Exporting.